![The Transformer Architecture](image-36.png)

## Multi-Head Attention

### 1. Why is it called Multi-Head Attention?

A single attention head calculates relationships between tokens using Query ($Q$), Key ($K$), and Value ($V$) matrices.

$$Q = XW_Q, \quad K = XW_K, \quad V = XW_V$$

Each head calculates attention:

$$\text{Head}_h = \text{softmax}\left(\frac{Q_h K_h^\top}{\sqrt{d_k}}\right)V_h$$

We use `multiple heads because different heads can learn different relationships` between tokens in `parallel`. For example, one head might learn pronoun relationships, while another might learn relationships between actions and objects. These are learned patterns, not fixed assignments.

### Why not use just one head?

Because one head has one learned set of query, key, and value projections. Multiple heads provide multiple independently learned attention patterns.

For the original Transformer architecture, the outputs from all heads are concatenated and projected:

$$\text{MultiHead}(X) = \text{Concat}(\text{Head}_1, \dots, \text{Head}_H)W_O$$

Here, $H$ is the number of attention heads, and $W_O$ is another learnable weight matrix.

For example, with $d_{\text{model}} = 768$ and 12 heads:

$$d_k = d_v = \frac{768}{12} = 64$$

Each head works with 64-dimensional query, key, and value representations. Their 12 outputs are concatenated to produce 768 features per token, before the final output projection preserves the model's hidden dimension.

### 3. What is the final output of Encoder (\(N\times\))?

This is the most important part.

In the original Transformer diagram, **\(N\times\)** means the encoder layer is repeated \(N\) times. Each layer contains:

1. Multi-head self-attention.
2. A residual connection and layer normalization (`Add & Norm`).
3. A feed-forward neural network.
4. Another residual connection and layer normalization.

The output of one encoder layer becomes the input to the next encoder layer.

![output of one encoder layer](image-33.png)

# Let discuss about the encode shape:

The output is not merely an attention-score matrix. It is a sequence of contextual token representations.

For a batch of sentences, the output shape is:

$$(B, L, d_{\text{model}})$$

where:

* $B$ = batch size.
* $L$ = padded sequence length.
* $d_{\text{model}}$ = hidden dimension.

Example: For your DistilBERT model, if you provide one sentence containing `nine` tokens, the final hidden state has shape:

torch.Size([1, 9, 768])

`DistilBERT` uses `six` Transformer encoder layers and `12 attention heads` per layer.

Each of the `nine` token positions receives a `768-dimensional` contextual representation. The final representation of bank, for example, can incorporate information from river and flooded.

#### Q. In transformer architecture I got this point the shape of multiheaded attentions is $X_N\in\mathbb{R}^{n\times d_{model}}$. Tell me how or a batch of sentences, the output shape is: 

#### $$(B,L,d_{\text{model}})$$
#### shape mapped with encoder output. Just explain shape change with batch added?

The original shape of multi-head attention for **one sentence** is

$$X_N \in \mathbb{R}^{N \times d_{\text{model}}}$$

When we process a **batch of sentences**, we add one extra dimension, $B$, to represent the number of sentences in the batch.

### Shape change

* $B$ = number of sentences in a batch
* $L$ = number of tokens in each sentence (sequence length)
* $d_{\text{model}}$ = number of features for each token

For one sentence:

$$X_N \in \mathbb{R}^{L \times d_{\text{model}}}$$

For a batch of sentences:

$$X_N \in \mathbb{R}^{B \times L \times d_{\text{model}}}$$

The only change is adding the batch dimension $B$ at the beginning. The encoder processes each sentence in the batch, producing its contextual representation while preserving the sequence length and model dimension.

For example, if $B = 32$, $L = 10$, and $d_{\text{model}} = 512$, the encoder output shape is

$$\boxed{(32, 10, 512)}$$
$$\downarrow$$

This means **32 sentences $\times$ 10 token positions $\times$ 512 features per token**.


#### L = number of token, this token assignee by models. Because user just pass text string to models and tokenization's done by models. I can understand sentence each assign to token ID. "Ram go school" required only 3 token + header & footer + padding. So token length depend upon each pass test string. Correct?

Yes, your understanding is correct. Let me clarify two important points.

1. **Tokenization**: The model's tokenizer converts the input text into token IDs. A token may be a whole word, part of a word, or a punctuation mark.
2. **Sequence length $L$**: It depends on the number of tokens produced by tokenization, plus any special tokens and padding added.

For example, suppose the text is `"Ram go school"`.

| Step | Example | Length |
| :--- | :--- | :---: |
| Input text | `Ram go school` | — |
| Tokenization | `[Ram, go, school]` | 3 |
| Add special tokens | `[BOS, Ram, go, school, EOS]` | 5 |
| Add padding (if needed) | `[BOS, Ram, go, school, EOS, PAD, PAD]` | 7 |

*Note: BOS and EOS are illustrative special tokens. The actual tokenizer may use different tokens or no such tokens.*

Therefore, the input shape for one sentence might be

$X \in \mathbb{R}^{L \times d_{\text{model}}}$

For a batch of sentences, it becomes

$\boxed{X \in \mathbb{R}^{B \times L \times d_{\text{model}}}}$

## Combine understanding Encoder + Multi Head dimensions

#### Q. Now I got encoder output is $X_N\in\mathbb{R}^{B\times L\times d_{\text{model}}}$ So If $d_{models} = 512$ then how the multiheaded output dimensions fit with this shape? Because of multiheaded attentions = 512/(total number of head.) another doubt, as per my understanding encoder output shape must be affected by multi head shape. But in your encoder shape dimensions their is not mentions for anything about mutli head attentions shape.

The number of heads affects the **internal dimensions** of multi-head attention:

$$\underbrace{512}_{d_{\text{model}}} = \underbrace{8}_{h} \times \underbrace{64}_{d_k}$$

But it does not normally change the final encoder output shape. `See below for how`

**Remember:** The number of heads controls how the attention computation is divided internally; $d_{\text{model}}$ controls the size of the representation passed between encoder blocks.


The key point is that **multi-head attention splits the feature dimension internally, but combines all heads back into $d_{\text{model}}$ before producing the encoder output.**

Let's use $d_{\text{model}} = 512$ and $h = 8$ attention heads.

### 1. How the 8 heads fit into 512 dimensions

The input to the multi-head attention layer has shape:

$$X \in \mathbb{R}^{B \times L \times 512}$$

The model divides the 512-dimensional representation across 8 heads.

$$d_k = d_v = \frac{d_{\text{model}}}{h} = \frac{512}{8} = 64$$

![Inside multi-head attention](image-34.png)

#### So number of head decided by whom in BERT and DistelBERT?

The number of attention heads is decided by the model's architecture designers when defining the Transformer model. It is a hyperparameter, not something the tokenizer decides from the input text.

For example, if $d_{\text{model}} = 768$ and the model uses 12 heads, each head has $768 / 12 = 64$ dimensions.

The number of heads remains fixed for that model, regardless of whether you pass a short sentence or a long paragraph.

### BERT vs. DistilBERT

For the standard Base versions, both use 12 attention heads, but they differ in the number of encoder layers.

| Configuration | BERT Base | DistilBERT Base |
| :--- | :---: | :---: |
| Model dimension $d_{\text{model}}$ | 768 | 768 |
| Number of heads $h$ | 12 | 12 |
| Dimensions per head | $768 / 12 = 64$ | $768 / 12 = 64$ |
| Encoder layers | 12 | 6 |

Sources: official model configurations for BERT Base and DistilBERT Base.

### 2. Who chooses these numbers?

The model designers choose the architecture's hyperparameters before training:

* $d_{\text{model}}$: the feature dimension.
* $h$: the number of attention heads.
* $N$: the number of encoder layers.

The dimensions must be compatible. For standard multi-head attention:

$$d_k = \frac{d_{\text{model}}}{h}$$

For both models:

$$\frac{768}{12} = 64$$

**Remember:** DistilBERT has fewer encoder layers than BERT Base, but it retains the same number of attention heads per layer. This is one way the distilled model reduces computational cost while preserving the same hidden representation size.


## Decoders

#### Q. Now I want to understand Decoder part. From encoder number of attention heads output provided to Decoder. Why the same input (Embedding + Positiona Encoding) given to decoder. We have already perform this steps in Encoders? What is meaning of output shifted right? Where this come form? 

The most important distinction is that the encoder reads the input sentence, while the decoder generates the output sentence. They do different jobs, so they have different inputs.

#### Why does the decoder have its own Embedding + Positional Encoding?
Imagine translating English into French:

* **Encoder input**: `I love learning`
* **Decoder input**: the target sentence's tokens that are already available, such as `J'aime`
* **Decoder output**: a probability distribution predicting the next target token, such as `apprendre`

### Why does the decoder need its own embedding?

1. The encoder embeds the **source-language tokens**.
2. The decoder embeds the **target-language tokens**.
3. Positional encoding gives the decoder information about the positions of those target tokens.

Even if both sides use the same model dimension, such as \(d_{\text{model}} = 512\), they process different token sequences.

The encoder's output is not fed into the decoder's embedding layer. Instead, it is passed to the decoder's **cross-attention layer**, which is the upper Multi-Head Attention block in your diagram.

#### What does "Outputs (shifted right)" mean?
TODO
#### Where does the shifted-right sequence come from?
TODO
### How does the decoder use the encoder output?

The decoder's two attention blocks have different purposes.

| Decoder block | What it attends to |
| :--- | :--- |
| Masked Multi-Head Attention | Previously available target tokens |
| Encoder–Decoder Multi-Head Attention (cross-attention) | Encoder output representations |

In cross-attention:

* **Queries ($Q$)** come from the decoder's current representations.
* **Keys ($K$) and Values ($V$)** come from the encoder output.

The decoder therefore uses both its target-side context and the source sentence's contextual representations to predict the next token.



#### Q. You have taken example of English to French. So positional numerical values is same for Encoder in English language and French language?

Yes and no. The positional encoding values depend on the token's position, not on whether the sentence is English or French. However, the encoder and decoder can use different positional encoding values depending on the Transformer implementation.

- Positional encoding: represents where a token occurs in the sequence.
- Token embedding: represents the token itself.
- Combined representation: contains both kinds of information.

import torch
from transformers import BertTokenizer, BertModel

In [1]:
import torch
from transformers import BertTokenizer, BertModel
from transformers import BertForPreTraining
import os
from dotenv import load_dotenv
from huggingface_hub import login

In [2]:
# 1. Load Hugging Face token from .env
load_dotenv()
token = os.getenv("HF_TOKEN")

# 2. Authenticate if a token is available
if token:
    login(token=token)
else:
    print("HF_TOKEN not found. Loading public model without authentication.")

# 3. Select device
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# 4. Define model name
model_name = "google-bert/bert-base-uncased"

# 5. Load tokenizer
tokenizer = BertTokenizer.from_pretrained(
    model_name,
    token=token or None
)

# 6. Load BERT pretraining model
model = BertForPreTraining.from_pretrained(
    model_name,
    token=token or None
).to(device)

# 7. Set evaluation mode
model.eval()

# 8. Display results
print("Device:", device)
print("Model:", model_name)
print("Tokenizer and model loaded successfully.")


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Loading weights:   0%|          | 0/206 [00:00<?, ?it/s]

Device: cuda
Model: google-bert/bert-base-uncased
Tokenizer and model loaded successfully.


In [3]:
def learn_bert_shape(sentences):
    """
    Tokenize a batch of sentences and inspect BERT encoder output shapes.

    Args:
        sentences (list[str]): Batch of input sentences.

    Returns:
        dict: Encoder output and shape information.
    """

    # 1. Tokenization and padding
    inputs = tokenizer(
        sentences,
        padding=True,
        truncation=True,
        return_tensors="pt"
    )

    # 2. Move inputs to the same device as the model
    inputs = {
        name: tensor.to(device)
        for name, tensor in inputs.items()
    }

    print("\nInput IDs shape:", inputs["input_ids"].shape)
    print("Attention mask shape:", inputs["attention_mask"].shape)

    # 3. Run the BERT encoder
    with torch.inference_mode():
        outputs = model.bert(**inputs)

    # 4. Final encoder output: (B, L, d_model)
    encoder_output = outputs.last_hidden_state

    print("\nEncoder output shape:", encoder_output.shape)
    print("Encoder output device:", encoder_output.device)

    # 5. Inspect model architecture
    print("\nNumber of attention heads:",
          model.config.num_attention_heads)

    print("Head dimension:",
          model.config.hidden_size //
          model.config.num_attention_heads)

    print("Hidden size:", model.config.hidden_size)
    print("Encoder layers:", model.config.num_hidden_layers)

    # 6. Extract the [CLS] token representation
    cls_output = encoder_output[:, 0, :]

    print("\n[CLS] output shape:", cls_output.shape)

    # 7. Return results for further use
    return {
        "inputs": inputs,
        "encoder_output": encoder_output,
        "cls_output": cls_output,
        "batch_size": encoder_output.shape[0],
        "sequence_length": encoder_output.shape[1],
        "hidden_size": encoder_output.shape[2],
    }


In [4]:
def print_shape(result):
    """
    Print BERT tokenization details, encoder shapes,
    attention configuration, and CLS representation shape.
    """

    inputs = result["inputs"]
    input_ids = inputs["input_ids"]
    attention_mask = inputs["attention_mask"]
    encoder_output = result["encoder_output"]
    cls_output = result["cls_output"]

    B, L, D = encoder_output.shape

    print("=" * 55)
    print("             BERT ENCODER SHAPE REPORT")
    print("=" * 55)

    # 1. Input shapes
    print("\n1. INPUT SHAPES")
    print(f"Input IDs          : {tuple(input_ids.shape)}")
    print(f"Attention mask     : {tuple(attention_mask.shape)}")
    print(f"Batch size (B)     : {B}")
    print(f"Sequence length (L): {L}")
    print(f"Hidden size (D)    : {D}")

    # 2. Tokenization details
    print("\n2. TOKENIZATION")
    for i in range(B):
        ids = input_ids[i].detach().cpu().tolist()
        tokens = tokenizer.convert_ids_to_tokens(ids)
        mask = attention_mask[i].detach().cpu().tolist()

        print(f"\nSentence {i + 1}")
        print(f"Token IDs      : {ids}")
        print(f"Tokens         : {tokens}")
        print(f"Attention mask : {mask}")

    # 3. Attention configuration
    print("\n3. MULTI-HEAD ATTENTION")
    print(f"Number of heads     : {D // 64 if D == 768 else 'See model config'}")
    print(f"Head dimension      : {D // (D // 64) if D == 768 else 'See model config'}")

    # 4. Encoder output
    print("\n4. FINAL ENCODER OUTPUT")
    print(f"Shape               : {tuple(encoder_output.shape)}")
    print(f"Meaning              : (B, L, D) = ({B}, {L}, {D})")
    print(f"Device               : {encoder_output.device}")
    print(f"Data type            : {encoder_output.dtype}")

    # 5. CLS representation
    print("\n5. CLS REPRESENTATION")
    print(f"CLS output shape     : {tuple(cls_output.shape)}")
    print("Meaning              : (B, D)")

    # 6. Small numerical sample
    print("\n6. SAMPLE VALUES (first token, first 10 features)")
    print(encoder_output[0, 0, :10].detach().cpu())

    print("\n" + "=" * 55)
    print("Shape report completed.")
    print("=" * 55)


In [5]:
# 3. Batch of sentences
sentences = [
    "I love learning transformers.",
    "BERT uses multi head attention."
]

In [6]:
result = learn_bert_shape(sentences)
print_shape(result)


Input IDs shape: torch.Size([2, 8])
Attention mask shape: torch.Size([2, 8])

Encoder output shape: torch.Size([2, 8, 768])
Encoder output device: cuda:0

Number of attention heads: 12
Head dimension: 64
Hidden size: 768
Encoder layers: 12

[CLS] output shape: torch.Size([2, 768])
             BERT ENCODER SHAPE REPORT

1. INPUT SHAPES
Input IDs          : (2, 8)
Attention mask     : (2, 8)
Batch size (B)     : 2
Sequence length (L): 8
Hidden size (D)    : 768

2. TOKENIZATION

Sentence 1
Token IDs      : [101, 1045, 2293, 4083, 19081, 1012, 102, 0]
Tokens         : ['[CLS]', 'i', 'love', 'learning', 'transformers', '.', '[SEP]', '[PAD]']
Attention mask : [1, 1, 1, 1, 1, 1, 1, 0]

Sentence 2
Token IDs      : [101, 14324, 3594, 4800, 2132, 3086, 1012, 102]
Tokens         : ['[CLS]', 'bert', 'uses', 'multi', 'head', 'attention', '.', '[SEP]']
Attention mask : [1, 1, 1, 1, 1, 1, 1, 1]

3. MULTI-HEAD ATTENTION
Number of heads     : 12
Head dimension      : 64

4. FINAL ENCODER OUTPUT
Shap

In [7]:
sentences = [
"Albert Einstein was a German-born theoretical physicist best known for developing the theory of relativity."
"Einstein also made important contributions to quantum theory." 
"His mass–energy equivalence formula E = mc², which arises from special relativity, has been called the world's most famous equation."
]

result = learn_bert_shape(sentences)
print_shape(result)


Input IDs shape: torch.Size([1, 57])
Attention mask shape: torch.Size([1, 57])

Encoder output shape: torch.Size([1, 57, 768])
Encoder output device: cuda:0

Number of attention heads: 12
Head dimension: 64
Hidden size: 768
Encoder layers: 12

[CLS] output shape: torch.Size([1, 768])
             BERT ENCODER SHAPE REPORT

1. INPUT SHAPES
Input IDs          : (1, 57)
Attention mask     : (1, 57)
Batch size (B)     : 1
Sequence length (L): 57
Hidden size (D)    : 768

2. TOKENIZATION

Sentence 1
Token IDs      : [101, 4789, 15313, 2001, 1037, 2446, 1011, 2141, 9373, 13702, 2190, 2124, 2005, 4975, 1996, 3399, 1997, 20805, 1012, 15313, 2036, 2081, 2590, 5857, 2000, 8559, 3399, 1012, 2010, 3742, 1516, 2943, 27841, 5675, 1041, 1027, 11338, 10701, 1010, 2029, 18653, 2013, 2569, 20805, 1010, 2038, 2042, 2170, 1996, 2088, 1005, 1055, 2087, 3297, 8522, 1012, 102]
Tokens         : ['[CLS]', 'albert', 'einstein', 'was', 'a', 'german', '-', 'born', 'theoretical', 'physicist', 'best', 'known', 'fo